# **MODELO DE CLASIFICACIÓN 2:**
# **Bosque Aleatorio (** Random Forest **)**

Este es un modelo de "ensamble", que se puede entender como **el siguiente paso después de Árbol de Decisión**. En lugar de crear un solo árbol (que puede ser propenso a "memorizar" los datos, u overfitting), un **Random Forest** crea cientos de árboles (`n_estimators=100`) y les pide que "voten" por la mejor clasificación.

**Es como pedir la opinión de 100 expertos en lugar de uno solo.** Son más potentes y robustos que un solo árbol. ¡Ventaja! Al igual que los árboles de decisión, los Random Forest no requieren que escalemos los datos.



****
## **Paso 0: Introducción y Librerías**

**Objetivo:** Entrenar un modelo que pueda predecir la especie de un pingüino basándose en sus medidas físicas (largo y profundidad del pico, largo de la aleta y masa corporal). Usaremos un **Árbol de Decisión**, un modelo fácil de interpretar.

In [ ]:
# Librerías para cargar el dataset y construir gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Librerías necesarias para el modelo DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Métricas de rendimiento
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

****
## **Paso 1: Cargar los datos**

In [ ]:
df = pd.read_csv('df_cleaned.csv')
df.head()

***
## **Paso 2: Cargar y Explorar los Datos**
## **Paso 3: Preparación de Datos (Limpieza de datos)**

Vamos a omitir estos pasos, ya que se realizaron durante la limpieza

***
## **Paso 4: Definir Features (X) y Target (y)**

Ahora, separamos nuestros datos en dos partes:

* **Features (`X`):** Las variables de entrada, las "preguntas" que le damos al modelo. Usaremos solo las medidas numéricas.

* **Target (`y`):** La variable de salida, la "respuesta" que queremos predecir (En este caso, las columna `species`).

In [ ]:
df.head()

* Definimos la columna (o lista de columnas) que será nuestro target `y`

In [ ]:
y = df['cut']     # y es una Serie (con un corchete)
y.head()

* Lista de columnas que usaremos como features y definimos `X`

In [ ]:
X = df[df.columns.drop('cut')] # X debe ser un DataFrame (con doble corchete)
X.head()

In [ ]:
# Mostramos las "clases" o especies únicas que el modelo debe aprender
print(f"Clases a predecir: {y.unique()}")

***
## **Paso 5: Dividir los datos en Train y Test**

Este es el paso más importante del Machine Learning. El set de datos se divide entre datos para entrenar el modelo y el resto para evaluarlo. Generalmente la relación entre datos de entrenamiento y evaluación es **80-20%** o **70-30%**.

* **Set de Entrenamiento (Train):** Lo usaremos para "entrenar" al modelo (80% de los datos).

*  **Set de Prueba (Test):** Lo guardaremos para "evaluar" al modelo con datos que nunca ha visto (20% de los datos).

* Usamos `random_state=42` para que la división aleatoria sea siempre la misma (reproducible).

In [ ]:
# Dividimos los datos (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total de datos de entrenamiento (train): {len(X_train)}")
print(f"Total de datos de prueba (test): {len(X_test)}")

***
## **Paso 6: Crear y Entrenar el Modelo**

Inicializamos el modelo **Random Forest**. Crearemos un <font color="red">**bosque de 100 árboles**</font> usando `n_estimators=100`.

* Entrenamos el modelo usando el método `.fit()` solo con los datos de entrenamiento.

In [ ]:
# 1. Inicializar el modelo
modelo_rf = RandomForestClassifier(max_depth=5,n_estimators=100, random_state=42)

# 2. Entrenar el modelo
modelo_rf.fit(X_train, y_train)

****
## **Paso 7: Evaluar el Modelo**

Ahora, usamos los datos de prueba (**X_test**) que el modelo nunca vio.

1. Hacemos predicciones con los datos de prueba usando `.predict()`

In [ ]:
y_pred = modelo_rf.predict(X_test)

2. Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Primero calculamos la **precisión** usando `.accuracy_score()`:

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy) del modelo: {accuracy * 100:.1f}%")

3. Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Ahora calculamos la **matriz de confusión** usando usando `.confusion_matrix()`:

In [ ]:
print("--- Matriz de Confusión ---")
print("\n(Las filas son el valor REAL, las columnas la PREDICCIÓN) \n")
labels = modelo_rf.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

# La mostramos como un DataFrame para que sea más fácil de leer
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

***
## **Paso 7.1: Optimizar los hiperparametros (`GridSearchCV`)**

GridSearchCV es la herramienta que permite que en lugar de probar "a mano" qué número de árboles o qué profundidad funciona mejor, dejas que Python lo haga por ti de forma sistemática.

>### **7.1.1. Definimos el diccionario de parámetros a probar y configuramos GridSearchCV**

* `cv=5` significa que usará Cross-Validation de 5 pliegues, es decir, entrena y prueba cada combinación de parámetros 5 veces.

* `n_jobs=-1` usa todos los núcleos de tu procesador para ir más rápido
* `verbose=1` muestra un resumen al inicio y al final

In [ ]:
parametros = {
    'n_estimators': [100, 200, 300, 500],        # Número de árboles en el bosque
    'max_depth': [3,5,10],          # Profundidad máxima de los árboles
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(estimator=rf, param_grid=parametros, cv=3, n_jobs=-1, verbose=1)

> ### **7.1.2. Entrenamos buscando la mejor combinación**

In [ ]:
%timeit
grid_search.fit(X_train, y_train)

> ### **7.1.3. Mostramos los resultados del modelo optimizado**

In [ ]:
print("--- Mejores Hiperparámetros encontrados ---")
print(grid_search.best_params_)

mejor_modelo = grid_search.best_estimator_

In [ ]:
# Accuracy del mejor modelo sobre el conjunto de prueba
accuracy_test = mejor_modelo.score(X_test, y_test)

print(f"Precisión (Accuracy) del modelo: {accuracy_test * 100:.1f}%")

***
## **Paso 8: Importancia de las Características (** Feature Importance **)**

* Obtenemos las importancias y creamos una serie de Pandas para verlas fácilmente


In [ ]:
importances = mejor_modelo.feature_importances_
feature_names = X_train.columns
forest_importances = pd.Series(importances, index=feature_names)

# Ordenamos de mayor a menor importancia
forest_importances_sorted = forest_importances.sort_values(ascending=False)

# Visualizamos la importancia de las características
plt.figure(figsize=(10, 6))
sns.barplot(x=forest_importances_sorted.values, y=forest_importances_sorted.index)
plt.title("Importancia de las Características en Random Forest")
plt.xlabel("Nivel de Importancia")
plt.ylabel("Característica")
plt.show()